In [1]:
!pip install datasets
!pip install transformers
!pip install bertviz

In [4]:
from datasets import load_dataset
from transformers import BertTokenizer, BertModel, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np
import random
from bertviz import head_view

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Task 1: Obtaining the dataset

In [12]:
train_dataset = load_dataset('ag_news', split = 'train')
test_dataset = load_dataset('ag_news', split = 'test')

#Task 2: Loading the pre-trained BERT

In [20]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model = model.to(device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

# Task 3: Experiments

# Task 3.1: Probing

In [22]:
def batch_get_embeddings(texts, tokenizer, model, strategy = "cls", max_length = 128, batch_size = 32):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc = f"Embedding ({strategy})"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(batch_texts, return_tensors = "pt", padding = True, truncation = True, max_length = max_length)
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)


        with torch.no_grad():
            outputs = model(input_ids = input_ids, attention_mask = attention_mask)

        hidden_states = outputs.last_hidden_state

        for j in range(hidden_states.size(0)):
            valid_tokens = hidden_states[j][attention_mask[j].bool()]

            if strategy == "cls":
                emb = hidden_states[j][0]

            elif strategy == "mean":
                emb = valid_tokens.mean(dim = 0)

            elif strategy == "last":
                emb = valid_tokens[-1]

            elif strategy == "first":
                emb = valid_tokens[0]

            else:
                raise ValueError(f"Unknown strategy: {strategy}")

            embeddings.append(emb.cpu().numpy())

    return embeddings

#"cls", "first", "last", "mean"
strategy = "mean"

#get texts and labels
X_train_texts = [ex['text'] for ex in train_dataset]
y_train = [ex['label'] for ex in train_dataset]

X_test_texts = [ex['text'] for ex in test_dataset]
y_test = [ex['label'] for ex in test_dataset]

#extract embeddings
X_train = batch_get_embeddings(X_train_texts, tokenizer, model, strategy = strategy)
X_test = batch_get_embeddings(X_test_texts, tokenizer, model, strategy = strategy)

#log regression
clf = LogisticRegression(max_iter = 1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
logreg_acc = accuracy_score(y_test, y_pred)
print(f"[Logistic Regression] Accuracy ({strategy} embedding): {logreg_acc:.4f}")

#knn
best_k = None
best_acc = 0
for k in [1, 3, 5, 7, 10]:
    knn = KNeighborsClassifier(n_neighbors = k)
    knn.fit(X_train, y_train)
    y_pred_knn = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred_knn)
    print(f"[KNN] Accuracy with k = {k} ({strategy} embedding): {acc:.4f}")
    if acc > best_acc:
        best_k = k
        best_acc = acc

print(f"[KNN] Best k = {best_k}, Accuracy = {best_acc:.4f} using {strategy} embeddings")

Embedding (mean): 100%|██████████| 238/238 [00:47<00:00,  5.03it/s]


[Logistic Regression] Accuracy (mean embedding): 0.9120
[KNN] Accuracy with k = 1 (mean embedding): 0.8978
[KNN] Accuracy with k = 3 (mean embedding): 0.9133
[KNN] Accuracy with k = 5 (mean embedding): 0.9188
[KNN] Accuracy with k = 7 (mean embedding): 0.9199
[KNN] Accuracy with k = 10 (mean embedding): 0.9188
[KNN] Best k = 7, Accuracy = 0.9199 using mean embeddings


## Task 3.2: Fine tuning

In [25]:
import os
os.environ["WANDB_DISABLED"] = "true"

dataset = load_dataset('ag_news')
dataset['train'] = dataset['train'].select(range(10000))

split_dataset = dataset['train'].train_test_split(test_size = 0.1, seed = 42)
dataset['train'] = split_dataset['train']
dataset['validation'] = split_dataset['test']
dataset['test'] = dataset['test'].select(range(1000))


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["text"], truncation = True, padding = "max_length", max_length = 128)

tokenized_datasets = dataset.map(tokenize_function, batched = True)
tokenized_datasets.set_format(type = 'torch', columns = ['input_ids', 'attention_mask', 'label'])


model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 4).to(device)

#for evluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)
    return {"accuracy": accuracy_score(labels, predictions)}


training_args = TrainingArguments(
    output_dir = "./results",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 64,
    num_train_epochs = 3,
    weight_decay = 0.01,
    logging_dir = "./logs",
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy",
    eval_steps = None,
)

#trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["validation"],
    compute_metrics = compute_metrics,
)

trainer.train()

eval_results = trainer.evaluate(tokenized_datasets["test"])
print(f"\nTest Accuracy: {eval_results['eval_accuracy']:.4f}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss,Accuracy
1,0.408600,0.270630,0.914000
2,0.190600,0.255475,0.925000
3,0.120200,0.291053,0.926000



Test Accuracy: 0.9180


## Task 3.4:

In [6]:
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_attentions=True)

# Example input
sentence = "Apple shares rose after the tech giant reported strong earnings."

inputs = tokenizer(sentence, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)

# Get attention from a specific layer
attention = outputs.attentions  # Tuple of (layer_count, batch, heads, seq_len, seq_len)

# Visualize using BertViz
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
head_view(attention, tokens)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


<IPython.core.display.Javascript object>